# Segmentación de Clientes

<img src="https://github.com/KarnikaKapoor/Files/blob/main/Colorful%20Handwritten%20About%20Me%20Blank%20Education%20Presentation.gif?raw=true">

En este proyecto realizaremos un agrupamiento (clustering) no supervisado de los registros de clientes de la base de datos de una tienda de abarrotes. La segmentación de clientes es la práctica de separar a los clientes en **grupos que reflejan similitudes entre los clientes de cada cluster**.

Dividiremos a los clientes en segmentos para **optimizar la relevancia de cada cliente para el negocio**.

Esto permite adaptar los productos según las necesidades y comportamientos distintos de los clientes. También ayuda al negocio a atender las inquietudes de los diferentes tipos de clientes.



<a id="1"></a>
## IMPORTACIÓN DE LIBRERÍAS

In [ ]:
#Importing the Libraries
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 100)
import datetime
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from yellowbrick.cluster import KElbowVisualizer
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt, numpy as np
from mpl_toolkits.mplot3d import Axes3D
from sklearn.cluster import AgglomerativeClustering
from matplotlib.colors import ListedColormap
from sklearn import metrics
import warnings
import sys
import plotly.express as px

#
if not sys.warnoptions:
    warnings.simplefilter("ignore")
#
np.random.seed(42)

## CARGA DE DATOS

In [ ]:
#Cargando el conjunto de datos
data = pd.read_csv("marketing_campaign.csv", sep="\t")
print("Número de datos:", len(data))
data.head()

<img src="https://github.com/KarnikaKapoor/Files/blob/main/Colorful%20Handwritten%20About%20Me%20Blank%20Education%20Presentation.png?raw=true">

Más información sobre los atributos:

**Personas**
* ID: identificador único del cliente
* Year_Birth: año de nacimiento del cliente
* Education: nivel educativo del cliente
* Marital_Status: estado civil del cliente
* Income: ingreso anual del hogar del cliente
* Kidhome: número de niños en el hogar del cliente
* Teenhome: número de adolescentes en el hogar del cliente
* Dt_Customer: fecha de inscripción del cliente con la empresa
* Recency: número de días desde la última compra del cliente
* Complain: 1 si el cliente se quejó en los últimos 2 años, 0 en caso contrario

**Productos**
* MntWines: monto gastado en vino en los últimos 2 años
* MntFruits: monto gastado en fruta en los últimos 2 años
* MntMeatProducts: monto gastado en carne en los últimos 2 años
* MntFishProducts: monto gastado en pescado en los últimos 2 años
* MntSweetProducts: monto gastado en dulces en los últimos 2 años
* MntGoldProds: monto gastado en productos "gold" en los últimos 2 años

**Promoción**
* NumDealsPurchases: número de compras hechas con descuento
* AcceptedCmp1: 1 si el cliente aceptó la oferta en la 1ª campaña, 0 en caso contrario
* AcceptedCmp2: 1 si el cliente aceptó la oferta en la 2ª campaña, 0 en caso contrario
* AcceptedCmp3: 1 si el cliente aceptó la oferta en la 3ª campaña, 0 en caso contrario
* AcceptedCmp4: 1 si el cliente aceptó la oferta en la 4ª campaña, 0 en caso contrario
* AcceptedCmp5: 1 si el cliente aceptó la oferta en la 5ª campaña, 0 en caso contrario
* Response: 1 si el cliente aceptó la oferta en la última campaña, 0 en caso contrario

**Lugar**
* NumWebPurchases: número de compras hechas a través del sitio web de la empresa
* NumCatalogPurchases: número de compras hechas usando un catálogo
* NumStorePurchases: número de compras hechas directamente en tienda
* NumWebVisitsMonth: número de visitas al sitio web de la empresa en el último mes

<a id="3"></a>
# <p style="background-color:#682F2F;font-family:newtimeroman;color:#FFF9ED;font-size:150%;text-align:center;border-radius:10px 10px;">LIMPIEZA DE DATOS</p>


**En esta sección**
* Limpieza de datos
* Ingeniería de características

Para tener una idea clara de qué pasos hay que seguir para limpiar el conjunto de datos, veamos primero la información general de `data`.


In [ ]:
#Información sobre las características
data.info()

In [ ]:
#Para eliminar los valores faltantes
data = data.dropna()
print("El número total de datos después de eliminar las filas con valores faltantes es:", len(data))

En el siguiente paso crearemos una característica a partir de **"Dt_Customer"** que indique el número de días que un cliente lleva registrado en la base de datos de la empresa. Para mantenerlo simple, tomaremos este valor en relación con el cliente más reciente del registro.

Por lo tanto, para obtener estos valores debemos revisar las fechas más nueva y más antigua registradas.

In [ ]:
#
data["Dt_Customer"]#.value_counts()

In [ ]:
#
pd.to_datetime(data["Dt_Customer"], dayfirst=True)

In [ ]:
#
data["Dt_Customer"] = pd.to_datetime(data["Dt_Customer"], dayfirst=True)

#
dates = []

#
for i in data["Dt_Customer"]:
    i = i.date()
    dates.append(i)

# Fechas de inscripción del cliente más nuevo y del más antiguo registrados
print("La fecha de inscripción del cliente más nuevo en los registros:", max(dates))
print("La fecha de inscripción del cliente más antiguo en los registros:", min(dates))


Creamos una característica **("Customer_For")** con el número de días que los clientes llevan comprando en la tienda, en relación con la última fecha registrada

In [ ]:
# Se creó la característica "Customer_For"
days = []

#
d1 = max(dates) #se toma como el cliente más nuevo

for i in dates:
    delta = (d1 - i).days
    days.append(delta)

data["Customer_For"] = days
data["Customer_For"] = pd.to_numeric(data["Customer_For"], errors="coerce")

data["Customer_For"]

Ahora exploraremos los valores únicos de las características categóricas para tener una idea clara de los datos.

In [ ]:
#
print("Total de categorías en la característica Marital_Status:\n", data["Marital_Status"].value_counts(), "\n")

In [ ]:
#
print("Total de categorías en la característica Education:\n", data["Education"].value_counts())

**A continuación, realizaremos los siguientes pasos para construir algunas características nuevas:**

* Extraer la **"Age"** (edad) del cliente a partir de **"Year_Birth"**, que indica el año de nacimiento de la persona.
* Crear una característica **"Spent"** que indique el monto total gastado por el cliente en las distintas categorías durante los últimos dos años.
* Crear una característica **"Living_With"** a partir de **"Marital_Status"** para extraer la situación de convivencia de las parejas.
* Crear una característica **"Children"** que indique el total de niños en un hogar, es decir, hijos e hijas más adolescentes.
* Para tener más claridad sobre el hogar, crear una característica que indique el **"Family_Size"** (tamaño de la familia).
* Crear una característica **"Is_Parent"** que indique el estatus de paternidad/maternidad.
* Por último, crearemos tres categorías en **"Education"** simplificando el conteo de sus valores.
* Eliminar algunas de las características redundantes.

In [ ]:
#Ingeniería de características
#Edad del cliente hoy
data["Age"] = 2021-data["Year_Birth"]

data["Age"]#.value_counts()

In [ ]:
#
plt.figure(figsize=(10, 6))
sns.histplot(data['Age'], bins=100, kde=True, color='maroon')
plt.title('Distribución de la edad (Age)')
plt.xlabel('Edad (Age)')
plt.ylabel('Frecuencia')
plt.show()

In [ ]:
#Gasto total en los distintos productos
data["Spent"] = data["MntWines"] + data["MntFruits"] + data["MntMeatProducts"] + \
                data["MntFishProducts"]+ data["MntSweetProducts"]+ data["MntGoldProds"]

data["Spent"]

In [ ]:
#Derivando la situación de convivencia a partir del estado civil ("Alone" = vive solo/a)
# Married     857
# Together    573
# Single      471
# Divorced    232
# Widow        76
# Alone         3
# Absurd        2
# YOLO          2

data["Living_With"] = data["Marital_Status"].replace({"Married":"Partner",
                                                      "Together":"Partner",
                                                      "Absurd":"Alone",
                                                      "Widow":"Alone",
                                                      "YOLO":"Alone",
                                                      "Divorced":"Alone",
                                                      "Single":"Alone",})

data["Living_With"].value_counts()

In [ ]:
#Característica que indica el total de niños/adolescentes en el hogar
data["Children"] = data["Kidhome"] + data["Teenhome"]

data["Children"].value_counts()

In [ ]:
#Característica para el total de integrantes del hogar
data["Family_Size"] = data["Living_With"].map({"Alone": 1, "Partner": 2}) + data["Children"]

data["Family_Size"].value_counts()

In [ ]:
#Característica relativa a la paternidad/maternidad
data["Is_Parent"] = np.where(data.Children> 0, 1, 0)

data["Is_Parent"].value_counts()

In [ ]:
#Segmentando los niveles educativos en tres grupos
# Graduation    1116
# PhD            481
# Master         365
# 2n Cycle       200
# Basic           54

data["Education"]= data["Education"].replace({"Basic":"Undergraduate",
                                              "2n Cycle":"Undergraduate",
                                              "Graduation":"Graduate",
                                              "Master":"Postgraduate",
                                              "PhD":"Postgraduate"})

data["Education"].value_counts()

In [ ]:
#Para mayor claridad
data = data.rename(columns={"MntWines": "Wines",
                            "MntFruits":"Fruits",
                            "MntMeatProducts":"Meat",
                            "MntFishProducts":"Fish",
                            "MntSweetProducts":"Sweets",
                            "MntGoldProds":"Gold"})


In [ ]:
#Eliminando algunas de las características redundantes
to_drop = ["Marital_Status", "Dt_Customer", "Z_CostContact", "Z_Revenue", "Year_Birth", "ID"]

data = data.drop(to_drop, axis=1)

Ahora que tenemos algunas características nuevas, veamos las estadísticas de los datos.

In [ ]:
#

data.describe()

**Graficando**

In [ ]:
#Para graficar algunas características seleccionadas
#Configurando las preferencias de color

sns.set(rc={"axes.facecolor":"#FFF9ED","figure.facecolor":"#FFF9ED"})

pallet = ["#682F2F", "#9E726F", "#D6B2B1", "#B9C0C9", "#9F8A78", "#F3AB60"]

cmap = colors.ListedColormap(["#682F2F", "#9E726F", "#D6B2B1", "#B9C0C9", "#9F8A78", "#F3AB60"])

#Graficando las siguientes características
To_Plot = [ "Income", "Recency", "Customer_For", "Age", "Spent", "Is_Parent"]

plt.figure()

sns.pairplot(data[To_Plot], hue= "Is_Parent",palette= (["#682F2F","#F3AB60"]))

plt.show()

Claramente hay algunos valores atípicos (outliers) en las características Income y Age.
Vamos a eliminar estos outliers de los datos.

In [ ]:
#Eliminando los outliers al poner un tope en Age e Income
data = data[(data["Age"]<90)]
data = data[(data["Income"]<150000)]
print("El número total de datos después de eliminar los outliers es:", len(data))

In [ ]:
#Graficando las siguientes características
To_Plot = [ "Income", "Recency", "Customer_For", "Age", "Spent", "Is_Parent"]

plt.figure()

sns.pairplot(data[To_Plot], hue= "Is_Parent",palette= (["#682F2F","#F3AB60"]))

plt.show()

A continuación, veamos la correlación entre las características.
(excluyendo por ahora los atributos categóricos)

In [ ]:
# Obtener la lista de variables categóricas (dtype object) antes de calcular la correlación
s_obj = (data.dtypes == 'object')
object_cols_for_corr = list(s_obj[s_obj].index)

# Crear un DataFrame temporal eliminando las columnas de tipo object para calcular la correlación
numeric_data_for_corr = data.drop(columns = object_cols_for_corr)

#matriz de correlación para los datos numéricos
corrmat= numeric_data_for_corr.corr()

plt.figure(figsize = (20,20))

sns.heatmap(corrmat, annot = True, cmap = cmap, center = 0, annot_kws={"size": 8})

print("Columnas omitidas del cálculo de correlación:", object_cols_for_corr)

Los datos ya están bastante limpios y se incluyeron las nuevas características. Continuaremos con el siguiente paso: el preprocesamiento de los datos.

<a id="4"></a>
# <p style="background-color:#682F2F;font-family:newtimeroman;color:#FFF9ED;font-size:150%;text-align:center;border-radius:10px 10px;">PREPROCESAMIENTO DE DATOS</p>

En esta sección preprocesaremos los datos para realizar las operaciones de clustering.

**Se aplican los siguientes pasos para preprocesar los datos:**

* Codificar (label encoding) las características categóricas
* Escalar las características usando el escalador estándar
* Crear un subconjunto del dataframe para la reducción de dimensionalidad

In [ ]:
#Obtener la lista de variables categóricas
s = (data.dtypes == 'object')
object_cols = list(s[s].index)

print("Variables categóricas en el conjunto de datos:", object_cols)

In [ ]:
#Codificando (Label Encoding) las variables de tipo object
LE = LabelEncoder()
for i in object_cols:
    data[i] = data[[i]].apply(LE.fit_transform)

print("Todas las características ahora son numéricas")

In [ ]:
#
data.Education.value_counts()

In [ ]:
#
data.Living_With.value_counts()

In [ ]:
#Creando una copia de los datos
ds = data.copy()

# creando un subconjunto del dataframe al eliminar las características de ofertas aceptadas y promociones
cols_del = ['AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1','AcceptedCmp2', 'Complain', 'Response']
ds = ds.drop(cols_del, axis = 1)

#Escalando
scaler = StandardScaler()
scaler.fit(ds)
scaled_ds = pd.DataFrame(scaler.transform(ds), columns = ds.columns)
print("Todas las características ahora están escaladas")

In [ ]:
#Datos escalados que se usarán para reducir la dimensionalidad
print("Dataframe que se usará para el modelado posterior:")
scaled_ds.head()

In [ ]:
#
ds.head()

In [ ]:
#
scaled_ds.describe().T

In [ ]:
#
ds.describe().T

<a id="5"></a>
# <p style="background-color:#682F2F;font-family:newtimeroman;color:#FFF9ED;font-size:150%;text-align:center;border-radius:10px 10px;">REDUCCIÓN DE DIMENSIONALIDAD</p>
En este problema hay muchos factores con base en los cuales se hará la clasificación final. Estos factores son, básicamente, atributos o características. Entre más características haya, más difícil es trabajar con ellas. Muchas de estas características están correlacionadas y, por lo tanto, son redundantes. Por eso realizaremos una reducción de dimensionalidad sobre las características seleccionadas antes de pasarlas por un clasificador.  
*La reducción de dimensionalidad es el proceso de reducir el número de variables aleatorias bajo consideración, obteniendo un conjunto de variables principales.*

El **análisis de componentes principales (PCA)** es una técnica para reducir la dimensionalidad de este tipo de conjuntos de datos, aumentando la interpretabilidad y, al mismo tiempo, minimizando la pérdida de información.

**Pasos en esta sección:**
* Reducción de dimensionalidad con PCA
* Graficar el dataframe reducido

**Reducción de dimensionalidad con PCA**

Para este proyecto, reduciremos las dimensiones a 3.

In [ ]:
#Iniciando PCA para reducir las dimensiones (características) a 3
pca = PCA(n_components=3)

pca.fit(scaled_ds)

PCA_ds = pd.DataFrame(pca.transform(scaled_ds), columns = (["col1","col2", "col3"]))

PCA_ds.describe().T

In [ ]:
# Calcular la varianza explicada por cada componente
explained_variance = pca.explained_variance_ratio_ * 100

fig = px.scatter_3d(PCA_ds, x='col1', y='col2', z='col3',
                    title='Proyección en 3D de los datos en la dimensión reducida',
                    color_continuous_scale=px.colors.sequential.Viridis,
                    size_max=2, # Ajustar el tamaño máximo de los marcadores
                    opacity=0.7)

fig.update_layout(
    scene=dict(
        xaxis_title=f'Componente Principal 1 ({explained_variance[0]:.2f}%)',
        yaxis_title=f'Componente Principal 2 ({explained_variance[1]:.2f}%)',
        zaxis_title=f'Componente Principal 3 ({explained_variance[2]:.2f}%)'
    ),
    title_font_size=20 # Ajustar el tamaño de fuente del título principal
)

fig.show()

<a id="6"></a>
# <p style="background-color:#682F2F;font-family:newtimeroman;color:#FFF9ED;font-size:150%;text-align:center;border-radius:10px 10px;">CLUSTERING</p>

Ahora que hemos reducido los atributos a tres dimensiones, realizaremos el clustering mediante Agglomerative Clustering (clustering jerárquico aglomerativo). Este método consiste en ir fusionando ejemplos hasta alcanzar el número deseado de clusters.

**Pasos para el clustering**
* Método del codo (Elbow Method) para determinar el número de clusters a formar
* Clustering mediante Agglomerative Clustering
* Examinar los clusters formados mediante un scatter plot

In [ ]:
# Revisión rápida del método del codo para encontrar el número de clusters a formar
print('Método del codo para determinar el número de clusters a formar:')
Elbow_M = KElbowVisualizer(KMeans(), k=10)

Elbow_M.fit(PCA_ds)

Elbow_M.show()

La celda anterior indica que cuatro es un número óptimo de clusters para estos datos.
A continuación, ajustaremos el modelo de Agglomerative Clustering para obtener los clusters finales.

In [ ]:
#Iniciando el modelo de Agglomerative Clustering
AC = AgglomerativeClustering(n_clusters=4)
# ajustar el modelo y predecir los clusters
yhat_AC = AC.fit_predict(PCA_ds)
PCA_ds["Clusters"] = yhat_AC
#Agregando la característica Clusters al dataframe original
data["Clusters"]= yhat_AC

Para examinar los clusters formados, veamos su distribución en 3D.

In [ ]:
#
fig = px.scatter_3d(PCA_ds, x='col1', y='col2', z='col3',
                    color=PCA_ds["Clusters"].astype(str), # Colorear por cluster, convertido a string para colores discretos
                    title='Gráfica de los clusters',
                    color_discrete_map={ # Definir un mapa de colores discreto usando 'pallet' o 'cmap' de matplotlib
                        '0': '#682F2F',
                        '1': '#B9C0C9',
                        '2': '#9F8A78',
                        '3': '#F3AB60'
                    },
                    size_max=1.5, # Ajustar el tamaño máximo de los marcadores
                    opacity=0.7)

fig.update_layout(
    scene=dict(
        xaxis_title='Componente Principal 1',
        yaxis_title='Componente Principal 2',
        zaxis_title='Componente Principal 3'
    ),
    title_font_size=20 # Ajustar el tamaño de fuente del título principal
)

fig.show()

## EVALUACIÓN DEL MODELO

Como este es un clustering no supervisado, no contamos con una etiqueta para evaluar o calificar el modelo. El propósito de esta sección es estudiar los patrones de los clusters formados y determinar la naturaleza de esos patrones.

Para eso, analizaremos los datos a la luz de los clusters mediante un análisis exploratorio y sacaremos conclusiones.

**Primero, veamos la distribución de los grupos del clustering**

In [ ]:
#Graficando el conteo de clusters
pal = ["#682F2F","#B9C0C9", "#9F8A78","#F3AB60"]
pl = sns.countplot(x=data["Clusters"], hue=data["Clusters"], palette=pal, legend=False)
pl.set_title("Distribución de los clusters")
plt.show()

Los clusters parecen estar bastante bien distribuidos.

In [ ]:
#
fig = px.scatter(data, x="Spent", y="Income", color=data["Clusters"].astype(str),
                 title="Perfil de los clusters según ingreso y gasto",
                 color_discrete_map={
                    '0': '#682F2F',
                    '1': '#B9C0C9',
                    '2': '#9F8A78',
                    '3': '#F3AB60'
                 },
                 hover_data=['Clusters', 'Spent', 'Income'])

fig.update_layout(title_font_size=20)
fig.show()

**La gráfica de ingreso vs. gasto muestra el patrón de los clusters**
* grupo 0: gasto alto e ingreso promedio
* grupo 1: gasto alto e ingreso alto
* grupo 2: gasto bajo e ingreso bajo
* grupo 3: gasto alto e ingreso bajo

A continuación, veremos la distribución detallada de los clusters según los distintos productos de los datos: vinos, frutas, carne, pescado, dulces y oro (Gold).

Exploremos ahora cómo les fue a nuestras campañas en el pasado.

In [ ]:
#Creando una característica con la suma de promociones aceptadas
data["Total_Promos"] = data["AcceptedCmp1"]+ data["AcceptedCmp2"]+ data["AcceptedCmp3"]+ data["AcceptedCmp4"]+ data["AcceptedCmp5"]
#Graficando el conteo de campañas totales aceptadas
plt.figure()
pl = sns.countplot(x=data["Total_Promos"],hue=data["Clusters"], palette= pal)
pl.set_title("Conteo de promociones aceptadas")
pl.set_xlabel("Número total de promociones aceptadas")
plt.show()

Hasta ahora no ha habido una respuesta abrumadora a las campañas. En general, muy pocos participantes. Además, nadie participó en las 5 campañas. Quizás se necesiten campañas mejor dirigidas y mejor planeadas para impulsar las ventas.


In [ ]:
#para más detalle sobre el estilo de compra
Places =["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases",  "NumWebVisitsMonth"]

for i in Places:
    plt.figure()
    sns.jointplot(x=data[i],y = data["Spent"],hue=data["Clusters"], palette= pal)
    plt.show()

<a id="8"></a>
# <p style="background-color:#682F2F;font-family:newtimeroman;color:#FFF9ED;font-size:150%;text-align:center;border-radius:10px 10px;">PERFILAMIENTO</p>

Ahora que hemos formado los clusters y revisado sus hábitos de compra, veamos quiénes están en cada uno de ellos. Para eso, perfilaremos los clusters formados y llegaremos a una conclusión sobre quién es nuestro cliente estrella y quién necesita más atención del equipo de marketing de la tienda.

Para decidir esto, graficaremos algunas de las características que son indicativas de los rasgos personales del cliente, a la luz del cluster al que pertenece.
Con base en los resultados, llegaremos a las conclusiones.

In [ ]:
Personal = [ "Kidhome","Teenhome","Customer_For", "Age", "Children", "Family_Size", "Is_Parent", "Education","Living_With"]

for i in Personal:
    plt.figure()
    sns.jointplot(x=data[i], y=data["Spent"], hue =data["Clusters"], kind="kde", palette=pal)
    plt.show()


**Puntos a destacar:**

La siguiente información se puede deducir sobre los clientes de los distintos clusters.

<img src="https://github.com/KarnikaKapoor/Files/blob/main/Colorful%20Handwritten%20About%20Me%20Blank%20Education%20Presentation%20(3).png?raw=true">
  

## CONCLUSIÓN

En este proyecto realicé un clustering no supervisado.
Utilicé reducción de dimensionalidad seguida de clustering aglomerativo.
Obtuve 4 clusters y los usé para perfilar a los clientes según su estructura familiar e ingreso/gasto.
Esto puede usarse para planear mejores estrategias de marketing.
